# Stage 1 -- Data, label structure and split protocols (E0)
Loads DIVE (Kaggle dataset `dive-smart-contracts`, or Zenodo fallback), verifies the
schema, computes the label statistics and co-occurrence matrix used by LGC, builds the
three split protocols (S1 random / S2 compiler-shift / S3 temporal) and publishes them
for all later stages. Outputs: results/, plots/ (PDF+PNG), logs/, tables/, splits.csv.
**Attach:** `scvd-lib`, `dive-smart-contracts` (optional). **Accelerator:** none needed.

In [3]:
import os, sys, json, time
import numpy as np, pandas as pd
LIB = '/kaggle/input/datasets/smnihalahmed/scvd-lib'
if not os.path.isdir(LIB):
    LIB = os.environ.get('SCVD_LIB', os.path.abspath('..'))   # local fallback
sys.path.insert(0, LIB)
import scvd
root, log = scvd.setup_stage('stage1_eda')
SEEDS = (0,) if scvd.quick() else scvd.RNG_SEEDS
log.info('seeds=%s quick=%s', SEEDS, scvd.quick())

2026-09-24 11:13:51,457 INFO stage stage1_eda initialised at /kaggle/working/stage1_eda
2026-09-24 11:13:51,459 INFO seeds=(0, 1, 2) quick=False


In [4]:
pre, post, lab, tools = scvd.load_dive(log)
fam = scvd.family_columns(pre, post)
for k, v in fam.items():
    log.info('family %-18s %3d features', k, len(v))
for v in scvd.VIEWS:
    log.info('view F_%-4s %3d features', v, len(scvd.view_columns(fam, v)))

2026-09-24 11:13:54,380 INFO loaded DIVE: pre=(22330, 222) post=(22330, 177) labels=(22330, 9) tools=(26071, 49)
2026-09-24 11:13:54,382 INFO family code_metadata        6 features
2026-09-24 11:13:54,382 INFO family interface_source    46 features
2026-09-24 11:13:54,383 INFO family creation_opcodes   168 features
2026-09-24 11:13:54,384 INFO family tx_metadata         10 features
2026-09-24 11:13:54,384 INFO family temporal_position    8 features
2026-09-24 11:13:54,385 INFO family runtime_opcodes    158 features
2026-09-24 11:13:54,386 INFO view F_pre  220 features
2026-09-24 11:13:54,387 INFO view F_post 176 features
2026-09-24 11:13:54,387 INFO view F_all  396 features


In [5]:
card = lab[scvd.LABELS].sum(1)
stats = pd.DataFrame({'label': scvd.LABELS, 'short': scvd.LABEL_SHORT,
                      'positive': [int(lab[c].sum()) for c in scvd.LABELS],
                      'rate': [lab[c].mean() for c in scvd.LABELS]})
stats['labelset_size_mean'] = card.mean(); stats['density'] = card.mean() / 8
stats.to_csv(f'{root}/results/stage1_label_stats.csv', index=False)
open(f'{root}/tables/stage1_label_stats.tex', 'w').write(
    stats[['short', 'positive', 'rate']].to_latex(index=False, float_format='%.4f'))
log.info('label cardinality=%.3f density=%.4f', card.mean(), card.mean() / 8)
stats

2026-09-24 11:13:57,059 INFO label cardinality=2.459 density=0.3074


,label,short,positive,rate,labelset_size_mean,density
0,Reentrancy,Reent,11400,0.510524,2.459427,0.307428
1,Access Control,AccCtrl,16723,0.748903,2.459427,0.307428
2,Arithmetic,Arith,9542,0.427318,2.459427,0.307428
3,Unchecked Return Values,UnchkRet,5911,0.264711,2.459427,0.307428
4,DoS,DoS,3781,0.169324,2.459427,0.307428
5,Bad Randomness,BadRnd,634,0.028392,2.459427,0.307428
6,Front Running,FrontR,606,0.027138,2.459427,0.307428
7,Time manipulation,TimeMan,6322,0.283117,2.459427,0.307428


In [6]:
scvd.plot_label_prevalence(lab, root, log)
M = scvd.plot_cooccurrence(lab, root, log)
pd.DataFrame(M, index=scvd.LABEL_SHORT, columns=scvd.LABEL_SHORT).to_csv(
    f'{root}/results/stage1_cooccurrence.csv')

2026-09-24 11:13:59,591 INFO maxp pruned
2026-09-24 11:13:59,604 INFO cmap pruned
2026-09-24 11:13:59,611 INFO kern dropped
2026-09-24 11:13:59,611 INFO post pruned
2026-09-24 11:13:59,614 INFO FFTM dropped
2026-09-24 11:13:59,617 INFO GPOS pruned
2026-09-24 11:13:59,619 INFO GSUB pruned
2026-09-24 11:13:59,624 INFO glyf pruned
2026-09-24 11:13:59,625 INFO Added gid0 to subset
2026-09-24 11:13:59,626 INFO Added first four glyphs to subset
2026-09-24 11:13:59,627 INFO Closing glyph list over 'MATH': 45 glyphs before
2026-09-24 11:13:59,628 INFO Glyph names: ['.notdef', '.null', 'A', 'B', 'C', 'D', 'F', 'M', 'R', 'S', 'T', 'U', 'a', 'c', 'comma', 'd', 'e', 'eight', 'five', 'four', 'h', 'i', 'k', 'l', 'm', 'n', 'nine', 'nonmarkingreturn', 'o', 'one', 'p', 'parenleft', 'parenright', 'percent', 'period', 'r', 's', 'seven', 'six', 'space', 't', 'three', 'two', 'v', 'zero']
2026-09-24 11:13:59,630 INFO Glyph IDs:   [0, 1, 2, 3, 8, 11, 12, 15, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 36, 37

In [7]:
splits = scvd.build_splits(pre, lab, seed=0)
p = scvd.save_splits(root, splits, pre, lab)
sizes = {s: {'train': len(v[0]), 'val': len(v[1]), 'test': len(v[2])} for s, v in splits.items()}
json.dump(sizes, open(f'{root}/results/stage1_split_sizes.json', 'w'), indent=1)
scvd.plot_split_sizes(sizes, root, log)
log.info('split sizes %s', sizes)

2026-09-24 11:14:02,969 INFO maxp pruned
2026-09-24 11:14:02,975 INFO cmap pruned
2026-09-24 11:14:02,976 INFO kern dropped
2026-09-24 11:14:02,977 INFO post pruned
2026-09-24 11:14:02,978 INFO FFTM dropped
2026-09-24 11:14:02,980 INFO GPOS pruned
2026-09-24 11:14:02,982 INFO GSUB pruned
2026-09-24 11:14:02,987 INFO glyf pruned
2026-09-24 11:14:02,988 INFO Added gid0 to subset
2026-09-24 11:14:02,988 INFO Added first four glyphs to subset
2026-09-24 11:14:02,989 INFO Closing glyph list over 'MATH': 21 glyphs before
2026-09-24 11:14:02,990 INFO Glyph names: ['.notdef', '.null', 'S', 'a', 'c', 'e', 'four', 'i', 'l', 'n', 'nonmarkingreturn', 'o', 'one', 'r', 's', 'space', 't', 'three', 'two', 'v', 'zero']
2026-09-24 11:14:02,991 INFO Glyph IDs:   [0, 1, 2, 3, 19, 20, 21, 22, 23, 54, 68, 70, 72, 76, 79, 81, 82, 85, 86, 87, 89]
2026-09-24 11:14:02,992 INFO Closed glyph list over 'MATH': 21 glyphs after
2026-09-24 11:14:02,992 INFO Glyph names: ['.notdef', '.null', 'S', 'a', 'c', 'e', 'four'

In [9]:
def push_dataset(root, stage, slug, title, log=None):
    """
    Push (or create) a Kaggle dataset from `root`.
    - If dataset already exists -> creates a new version.
    - If it doesn't exist yet -> initializes metadata and creates it.
    Returns the dataset URL, or None if no credentials/API available.
    """
    import os, json
    from kaggle.api.kaggle_api_extended import KaggleApi

    try:
        api = KaggleApi()
        api.authenticate()
    except Exception as e:
        if log: log.info("no kaggle credentials available (%s); skipping push", e)
        return None

    # Kaggle hard limits: slug 6-50 chars, title 6-50 chars
    if not (6 <= len(slug) <= 50):
        raise ValueError(f"slug must be 6-50 chars, got {len(slug)}: {slug!r}")
    if len(title) > 50:
        title = title[:50].rstrip()
        if log: log.info("title truncated to 50 chars: %r", title)
    if len(title) < 6:
        raise ValueError(f"title must be at least 6 chars, got {len(title)}: {title!r}")

    username = api.get_config_value('username')
    dataset_id = f"{username}/{slug}"
    url = f"https://www.kaggle.com/datasets/{dataset_id}"

    meta_path = os.path.join(root, 'dataset-metadata.json')

    if not os.path.exists(meta_path):
        api.dataset_initialize(root)

    with open(meta_path, 'r') as f:
        meta = json.load(f)
    meta['id'] = dataset_id
    meta['title'] = title
    with open(meta_path, 'w') as f:
        json.dump(meta, f, indent=2)

    try:
        api.dataset_create_version(
            root,
            f"update: {stage} run",
            quiet=False,
            dir_mode='zip'
        )
        if log: log.info("pushed new version -> %s", url)
    except Exception as e:
        if log: log.info("create_version failed (%s); creating new dataset instead", e)
        api.dataset_create_new(
            folder=root,
            quiet=False,
            dir_mode='zip',
            public=False
        )
        if log: log.info("created dataset -> %s", url)

    return url


title = 'SCVD Stage 1: DIVE EDA, label stats & splits'  # 45 chars, under the 50 cap
url = push_dataset(root, 'stage1_eda', 'scvd-stage1-eda', title, log)
if url:
    print('DATASET:', url)
else:
    print('push skipped (no credentials); artifacts remain under', root)

2026-09-24 11:41:01,579 INFO no kaggle credentials available (HTTPSConnectionPool(host='api.kaggle.com', port=443): Max retries exceeded with url: /v1/security.OAuthService/IntrospectToken (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7f3ab38c7740>: Failed to resolve 'api.kaggle.com' ([Errno -3] Temporary failure in name resolution)"))); skipping push
push skipped (no credentials); artifacts remain under /kaggle/working/stage1_eda
